# DQN on Atari Pong — Training

Reproduction of **Mnih et al., *Human-level control through deep reinforcement learning* (Nature, 2015)** on the Atari game **Pong**, using PyTorch and Gymnasium.

This notebook calls the modular code in `src/` so the training logic stays in one place. Run it on **Google Colab with a GPU** (T4 or better).

## 1. Install dependencies (Colab)

In [ ]:
!pip install -q -U "gymnasium[atari]" "autorom[accept-rom-license]" ale-py imageio
!AutoROM --accept-license -q || true

## 2. Clone the repo and import the modules

If you are running this locally instead of Colab, just make sure the working directory is the repo root.

In [ ]:
import sys, os
# On Colab, clone the repo:
# !git clone https://github.com/<your-username>/DQN-Pong.git
# %cd DQN-Pong
sys.path.append(os.getcwd())

## 3. Sanity check on the environment

In [ ]:
import numpy as np, torch
from src.env import make_env
from src.model import DQN
from src.config import DEVICE, SEED

env = make_env()
s, _ = env.reset(seed=SEED)
assert env.observation_space.shape == (4, 84, 84)
assert np.asarray(s).dtype == np.uint8
dummy = torch.zeros(2, 4, 84, 84, dtype=torch.uint8, device=DEVICE)
assert DQN(env.action_space.n).to(DEVICE)(dummy).shape == (2, env.action_space.n)
env.close()
print("Sanity check OK:", np.asarray(s).shape, np.asarray(s).dtype, "| device:", DEVICE)

## 4. Train

Training runs up to `TOTAL_STEPS = 1.5M` environment steps with **early stopping** when the average reward over the last 100 episodes reaches `TARGET_REWARD = 18.0`. On a Colab T4 this typically takes **3–5 hours**.

In [ ]:
from src.train import train
agent, returns, history = train()

## 5. Save artifacts

Persist the training history (returns + epoch-level metrics) so the evaluation notebook can reproduce the plots without retraining.

In [ ]:
import json, os
os.makedirs("artifacts", exist_ok=True)
with open("artifacts/history.json", "w") as f:
    json.dump({"returns": list(map(float, returns)), "history": history}, f)
print("saved artifacts/history.json")